# CRISP-DM Phase 4: Modelling — Model 2, Track A
## SVM Baseline — Selected Features (k=7)
Loads the pre-selected feature sets produced by feature-selection.ipynb.
Pipeline: SelectKBest output → StandardScaler → SVC (default params)
NOTE: The SVM baseline on k=7 features is expected to perform poorly.
This is a methodological finding — if F1 approaches zero, it indicates
the k=7 feature subset is insufficient for SVM's distance-based boundary.
Chi-squared selects on univariate signal; SVM requires multivariate
spatial separation across all dimensions simultaneously.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# Load pre-selected feature sets from feature-selection.ipynb
X_train = joblib.load('../data/X_train_sel.pkl')
X_test  = joblib.load('../data/X_test_sel.pkl')
y_train = joblib.load('../data/y_train.pkl')
y_test  = joblib.load('../data/y_test.pkl')

X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

Train: (4800, 7) | Test: (1200, 7)
Features: ['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']


## Standardisation — Per Column, Fit on Training Data Only
SVM is a distance-based algorithm — standardisation is mandatory.
Without scaling, features with large magnitudes (e.g. income) dominate
the hyperplane calculation, rendering other features effectively invisible.
Fitted on training data only; transform applied to both sets.

In [2]:
continuous_cols = [col for col in X_train.columns
                   if col in ['age', 'yrs_experience', 'family_size',
                               'income', 'mortgage_amt', 'credit_card_spend']]

scaler = StandardScaler()

for col in continuous_cols:
    scaler.fit(X_train[col].values.reshape(-1, 1))
    X_train[col] = scaler.transform(X_train[col].values.reshape(-1, 1))
    X_test[col]  = scaler.transform(X_test[col].values.reshape(-1, 1))

print(f"Scaled columns: {continuous_cols}")

Scaled columns: ['yrs_experience', 'income', 'mortgage_amt']


## SVM Baseline — Default Parameters
All parameters at sklearn defaults. random_state=42 for reproducibility.
Expected finding: if F1 is near zero, the k=7 chi2-selected features
provide insufficient spatial variance for SVM's hyperplane calculation.
Result feeds directly into champion model selection in tune-params.ipynb.

In [3]:
svm = SVC(random_state=42)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)
f1          = 2 * (precision * recall) / (precision + recall)
specificity = tn / (tn + fp)

print("=== SVM Baseline — Selected Features (k=7) ===")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn}  FP: {fp}")
print(f"  FN: {fn}  TP: {tp}")
print(f"\nPrecision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"\nFull classification report:")
print(classification_report(y_test, y_pred))
print("\nNOTE: F1 near zero = k=7 feature subset incompatible with SVM.")
print("This configuration will be excluded from hyperparameter tuning.")

=== SVM Baseline — Selected Features (k=7) ===

Confusion Matrix:
  TN: 976  FP: 44
  FN: 62  TP: 118

Precision:   0.7284
Recall:      0.6556
F1-Score:    0.6901
Specificity: 0.9569

Full classification report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      1020
           1       0.73      0.66      0.69       180

    accuracy                           0.91      1200
   macro avg       0.83      0.81      0.82      1200
weighted avg       0.91      0.91      0.91      1200


NOTE: F1 near zero = k=7 feature subset incompatible with SVM.
This configuration will be excluded from hyperparameter tuning.
